# Simple Signal Backtest

Run the first threshold signal-taker strategy.

## Contract-Aware Signal Backtest Design

The old signal used `forecast_above_threshold`, which is only valid for `>` contracts. The corrected research path uses `forecast_event_indicator`:

- `< threshold`: event is true when forecast temperature is below the threshold.
- `> threshold`: event is true when forecast temperature is above the threshold.
- `range`: event is true when forecast temperature is inside `[threshold_value, threshold_upper_value]`.

This matters because using a one-direction threshold feature can invert signals on lower-tail markets and create false edge.

In [ ]:
from pathlib import Path
import polars as pl

dataset_path = Path("../data/processed/datasets/weather_nyc_main_v1_features/part-0.parquet")
df = pl.read_parquet(dataset_path) if dataset_path.exists() else pl.DataFrame()
required = {"label", "market_ticker", "market_mid", "market_spread", "forecast_event_indicator"}
usable = df.drop_nulls(list(required - {"market_ticker"})) if required.issubset(df.columns) else pl.DataFrame()
print({
    "rows": df.height,
    "usable_signal_rows": usable.height,
    "markets": df.select("market_ticker").n_unique() if "market_ticker" in df.columns else 0,
    "usable_markets": usable.select("market_ticker").n_unique() if "market_ticker" in usable.columns else 0,
})

summary = (
    usable.with_columns(
        (pl.col("forecast_event_indicator") - pl.col("market_mid") / 100).alias("event_edge")
    )
    .with_columns(
        pl.when(pl.col("event_edge").abs() < 0.05).then(pl.lit("<5c"))
        .when(pl.col("event_edge").abs() < 0.10).then(pl.lit("5-10c"))
        .when(pl.col("event_edge").abs() < 0.20).then(pl.lit("10-20c"))
        .when(pl.col("event_edge").abs() < 0.40).then(pl.lit("20-40c"))
        .otherwise(pl.lit("40c+")).alias("edge_bucket")
    )
    .group_by("edge_bucket")
    .agg([
        pl.len().alias("rows"),
        pl.col("market_ticker").n_unique().alias("markets"),
        ((pl.col("label") == (pl.col("event_edge") > 0).cast(pl.Int64))).mean().alias("direction_hit_rate"),
        pl.col("market_spread").median().alias("median_spread_cents"),
    ])
    .sort("edge_bucket")
    if usable.height else pl.DataFrame({"status": [f"missing or unusable {dataset_path}"]})
)
display(summary)

### Backtest Recommendation

Implement the signal strategy as an executable edge strategy, not a midpoint toy strategy:

- Buy YES only if `p_model * 100 - yes_ask_cents > min_edge_cents + fee_buffer`.
- Buy NO only if `(1 - p_model) * 100 - no_ask_cents > min_edge_cents + fee_buffer`.
- Do not use midpoint fills for taker simulation.
- Require non-null bid/ask, spread, label, model probability, and event metadata.
- Cap exposure by event date because the six markets are mutually exclusive and highly correlated.

The initial `forecast_event_indicator` should be treated as a baseline feature. Production signals should use a calibrated probability model trained on contract-aware features.